In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
 
catalog = "workspace"
schema = "stock_analytics"
 
print("=" * 70)
print("ANALYTICS LAYER - GOLD TABLES")
print("=" * 70)

In [0]:

# Cell 1: Load Silver Layer
silver_df = spark.table(f"{catalog}.{schema}.silver_stock_indicators")
 
print("✅ Loaded silver_stock_indicators")

In [0]:

# Cell 2: Gold Table 1 - Current Trading Signals
print("\n" + "=" * 70)
print("CREATING GOLD TABLES")
print("=" * 70 + "\n")
 
# Get latest price for each stock
window_latest = Window.partitionBy("Ticker").orderBy(F.desc("Date"))
 
gold_signals = silver_df \
    .withColumn("rank", F.row_number().over(window_latest)) \
    .filter(F.col("rank") == 1) \
    .select(
        "Ticker", "Date", "Close", "Volume",
        "MA_20", "MA_50", "MA_200",
        "RSI_14", "MACD_Line", "MACD_Signal", "Volatility_30",
        "BB_Upper", "BB_Lower",
        "Price_Trend_20", "Price_Trend_200",
        "Golden_Cross", "Death_Cross"
    ) \
    .withColumn(
        "Trading_Signal",
        F.when(F.col("Golden_Cross") == "BUY_SIGNAL", "STRONG_BUY")
         .when(F.col("Death_Cross") == "SELL_SIGNAL", "STRONG_SELL")
         .when((F.col("RSI_14") < 30) & (F.col("Price_Trend_20") == "Bullish"), "BUY")
         .when((F.col("RSI_14") > 70) & (F.col("Price_Trend_20") == "Bearish"), "SELL")
         .when((F.col("MACD_Line") > F.col("MACD_Signal")) & (F.col("Price_Trend_20") == "Bullish"), "ACCUMULATE")
         .when((F.col("MACD_Line") < F.col("MACD_Signal")) & (F.col("Price_Trend_20") == "Bearish"), "DISTRIBUTE")
         .otherwise("HOLD")
    ) \
    .withColumn(
        "Confidence",
        F.round(
            (F.when(F.col("Price_Trend_20") == "Bullish", 1).otherwise(0) +
             F.when(F.col("Price_Trend_200") == "Uptrend", 1).otherwise(0) +
             F.when(F.col("RSI_14").between(30, 70), 1).otherwise(0) +
             F.when(F.col("MACD_Line") > F.col("MACD_Signal"), 1).otherwise(0)) / 4.0 * 100,
            2
        )
    )
 
gold_signals_table = f"{catalog}.{schema}.gold_trading_signals"
gold_signals.write.format("delta").mode("overwrite").saveAsTable(gold_signals_table)
 
print(f"✅ Created: {gold_signals_table}")

In [0]:

# Cell 3: Gold Table 2 - Stock Performance Rankings
print("\n" + "=" * 70)
print("STOCK PERFORMANCE RANKINGS")
print("=" * 70 + "\n")
 
# Calculate returns and rankings
gold_performance = silver_df \
    .withColumn("rank_date", F.row_number().over(Window.partitionBy("Ticker").orderBy(F.desc("Date")))) \
    .filter(F.col("rank_date") <= 1) \
    .select("Ticker", "Date", "Close", "Volume") \
    .join(
        silver_df.filter(F.col("Date") >= F.add_months(F.current_date(), -1))
            .withColumn("rank_date", F.row_number().over(Window.partitionBy("Ticker").orderBy("Date")))
            .filter(F.col("rank_date") == 1)
            .select(F.col("Ticker").alias("Ticker_1M"), F.col("Close").alias("Close_1M")),
        F.col("Ticker") == F.col("Ticker_1M"),
        "left"
    ) \
    .join(
        silver_df.filter(F.col("Date") >= F.add_months(F.current_date(), -3))
            .withColumn("rank_date", F.row_number().over(Window.partitionBy("Ticker").orderBy("Date")))
            .filter(F.col("rank_date") == 1)
            .select(F.col("Ticker").alias("Ticker_3M"), F.col("Close").alias("Close_3M")),
        F.col("Ticker") == F.col("Ticker_3M"),
        "left"
    ) \
    .select(
        "Ticker", "Date", "Close",
        F.col("Close_1M"), F.col("Close_3M")
    ) \
    .withColumn(
        "Return_1M_Percent",
        F.round(((F.col("Close") - F.col("Close_1M")) / F.col("Close_1M") * 100), 2)
    ) \
    .withColumn(
        "Return_3M_Percent",
        F.round(((F.col("Close") - F.col("Close_3M")) / F.col("Close_3M") * 100), 2)
    ) \
    .withColumn(
        "Performance_Rank",
        F.row_number().over(Window.orderBy(F.desc("Return_1M_Percent")))
    ) \
    .withColumn(
        "Performance_Tier",
        F.when(F.col("Performance_Rank") <= 3, "Top 3")
         .when(F.col("Performance_Rank") <= 5, "Top 5")
         .when(F.col("Performance_Rank") > 7, "Underperformer")
         .otherwise("Mid-tier")
    )
 
gold_performance_table = f"{catalog}.{schema}.gold_stock_performance"
gold_performance.write.format("delta").mode("overwrite").saveAsTable(gold_performance_table)
 
print(f"✅ Created: {gold_performance_table}")
 

In [0]:
# Cell 4: Gold Table 3 - Volatility Analysis
print("\n" + "=" * 70)
print("VOLATILITY ANALYSIS")
print("=" * 70 + "\n")

window_latest_vol = Window.partitionBy("Ticker").orderBy(F.desc("Date"))

gold_volatility = silver_df \
    .withColumn("rank", F.row_number().over(window_latest_vol)) \
    .filter(F.col("rank") == 1) \
    .select(
        "Ticker", "Date", "Close",
        "RSI_14", "Volatility_30", "BB_Upper", "BB_Lower"
    ) \
    .withColumn(
        "BB_Position",
        F.round((F.col("Close") - F.col("BB_Lower")) / (F.col("BB_Upper") - F.col("BB_Lower")) * 100, 2)
    ) \
    .withColumn(
        "Volatility_Status",
        F.when(F.col("RSI_14") > 70, "OVERBOUGHT")
         .when(F.col("RSI_14") < 30, "OVERSOLD")
         .otherwise("NEUTRAL")
    ) \
    .withColumn(
        "Bollinger_Alert",
        F.when(F.col("Close") >= F.col("BB_Upper"), "ABOVE_UPPER_BAND")
         .when(F.col("Close") <= F.col("BB_Lower"), "BELOW_LOWER_BAND")
         .otherwise("WITHIN_BANDS")
    )

gold_volatility_table = f"{catalog}.{schema}.gold_volatility_analysis"
gold_volatility.write.format("delta").mode("overwrite").saveAsTable(gold_volatility_table)

print(f"✅ Created: {gold_volatility_table}")